In [ ]:
#Loading dataset
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 100)

In [2]:
DES_PATH = "../data/processed/crop_yield/des_apy_2013_14_2022_23.csv"
UPAG_PATH = "../data/processed/crop_yield/upag_2022_23_2024_25.csv"

des = pd.read_csv(DES_PATH)
upag = pd.read_csv(UPAG_PATH)

print("DES shape:", des.shape)
print("UPAg shape:", upag.shape)

DES shape: (85920, 16)
UPAg shape: (9384, 16)


In [3]:
des_2022 = des[
    des["year"] == "2022-2023"
].copy()

upag_2022 = upag[
    upag["year"] == "2022-2023"
].copy()

print("DES 2022-23:", des_2022.shape)
print("UPAg 2022-23:", upag_2022.shape)

DES 2022-23: (9742, 16)
UPAg 2022-23: (3128, 16)


In [4]:
#Defining Crop season mapping
CROP_SEASON_MAP = {
    "Rice": "Kharif",
    "Wheat": "Rabi",
    "Maize": "Kharif",
    "Urad": "Kharif"
}

In [5]:
validation_crops = list(
    CROP_SEASON_MAP.keys()
)

print(CROP_SEASON_MAP)

{'Rice': 'Kharif', 'Wheat': 'Rabi', 'Maize': 'Kharif', 'Urad': 'Kharif'}


In [6]:
#Extracting comparable DES observations
def get_des_comparable_data(
    des_df,
    crop_season_map
):
    
    frames = []

    for crop, season in crop_season_map.items():

        temp = des_df[
            (des_df["crop"] == crop) &
            (des_df["season"] == season)
        ].copy()

        temp["comparison_season"] = season

        frames.append(temp)

    return pd.concat(
        frames,
        ignore_index=True
    )

In [7]:
des_comparable = get_des_comparable_data(
    des_2022,
    CROP_SEASON_MAP
)

print(
    "Comparable DES observations:",
    des_comparable.shape
)

display(
    des_comparable[
        [
            "crop",
            "season",
            "state",
            "district",
            "state_code",
            "district_code"
        ]
    ].head()
)

Comparable DES observations: (2062, 17)


,crop,season,state,district,state_code,district_code
0,Rice,Kharif,Andhra Pradesh,Alluri Sitharama Raju,28,745
1,Rice,Kharif,Andhra Pradesh,Anakapalli,28,744
2,Rice,Kharif,Andhra Pradesh,Ananthapuramu,28,502
3,Rice,Kharif,Andhra Pradesh,Annamayya,28,753
4,Rice,Kharif,Andhra Pradesh,Bapatla,28,750


In [8]:
#Checking coverage before merging
des_coverage = (
    des_comparable
    .groupby("crop")
    .agg(
        districts=(
            "district_code",
            "nunique"
        ),
        observations=(
            "district_code",
            "size"
        )
    )
    .reset_index()
)

upag_coverage = (
    upag_2022[
        upag_2022["crop"].isin(
            validation_crops
        )
    ]
    .groupby("crop")
    .agg(
        districts=(
            "district_code",
            "nunique"
        ),
        observations=(
            "district_code",
            "size"
        )
    )
    .reset_index()
)

coverage = des_coverage.merge(
    upag_coverage,
    on="crop",
    suffixes=(
        "_des",
        "_upag"
    )
)

display(coverage)

,crop,districts_des,observations_des,districts_upag,observations_upag
0,Maize,552,552,782,782
1,Rice,485,485,782,782
2,Urad,488,488,782,782
3,Wheat,537,537,782,782


In [11]:
#Merging DES and UPAG
comparison = pd.merge(
    des_comparable,
    upag_2022[
        upag_2022["crop"].isin(
            validation_crops
        )
    ],
    on=[
        "state_code",
        "district_code",
        "crop"
    ],
    how="outer",
    suffixes=(
        "_des",
        "_upag"
    ),
    indicator=True
)

print(
    comparison["_merge"]
    .value_counts()
)

_merge
both          2062
right_only    1066
left_only        0
Name: count, dtype: int64


In [12]:
#Calculating match percentage
match_summary = (
    comparison
    .groupby("crop")
    .agg(
        total_comparison_rows=(
            "_merge",
            "size"
        ),
        matched=(
            "_merge",
            lambda x: (x == "both").sum()
        ),
        des_only=(
            "_merge",
            lambda x: (x == "left_only").sum()
        ),
        upag_only=(
            "_merge",
            lambda x: (x == "right_only").sum()
        )
    )
    .reset_index()
)

match_summary["match_percentage"] = (
    match_summary["matched"]
    /
    (
        match_summary["matched"]
        +
        match_summary["des_only"]
    )
    * 100
)

display(match_summary)

,crop,total_comparison_rows,matched,des_only,upag_only,match_percentage
0,Maize,782,552,0,230,100.0
1,Rice,782,485,0,297,100.0
2,Urad,782,488,0,294,100.0
3,Wheat,782,537,0,245,100.0


In [13]:
#Keeping only matched observations
matched = comparison[
    comparison["_merge"] == "both"
].copy()

print(
    "Matched observations:",
    len(matched)
)

Matched observations: 2062


In [14]:
#Calculating  absolute differences
#Area
matched["area_difference"] = (
    matched["area_ha_des"]
    -
    matched["area_ha_upag"]
)

#Production
matched["production_difference"] = (
    matched["production_tonnes_des"]
    -
    matched["production_tonnes_upag"]
)

#Yield
matched["yield_difference"] = (
    matched["yield_kg_ha_des"]
    -
    matched["yield_kg_ha_upag"]
)

In [15]:
#Calculating percentage differences
#Area
matched["area_difference_pct"] = (
    matched["area_difference"].abs()
    /
    matched["area_ha_des"].abs()
    * 100
)

#Production
matched["production_difference_pct"] = (
    matched["production_difference"].abs()
    /
    matched["production_tonnes_des"].abs()
    * 100
)

#Yield
matched["yield_difference_pct"] = (
    matched["yield_difference"].abs()
    /
    matched["yield_kg_ha_des"].abs()
    * 100
)

In [16]:
#Fixing division by zero issue
matched.loc[
    matched["area_ha_des"] == 0,
    "area_difference_pct"
] = np.nan

matched.loc[
    matched["production_tonnes_des"] == 0,
    "production_difference_pct"
] = np.nan

matched.loc[
    matched["yield_kg_ha_des"] == 0,
    "yield_difference_pct"
] = np.nan

In [17]:
#Generating main validation report
validation_report = (
    matched
    .groupby("crop")
    .agg(
        matched_observations=(
            "crop",
            "size"
        ),

        mean_area_difference_pct=(
            "area_difference_pct",
            "mean"
        ),

        median_area_difference_pct=(
            "area_difference_pct",
            "median"
        ),

        mean_production_difference_pct=(
            "production_difference_pct",
            "mean"
        ),

        median_production_difference_pct=(
            "production_difference_pct",
            "median"
        ),

        mean_yield_difference_pct=(
            "yield_difference_pct",
            "mean"
        ),

        median_yield_difference_pct=(
            "yield_difference_pct",
            "median"
        )
    )
    .reset_index()
)

display(validation_report)

,crop,matched_observations,mean_area_difference_pct,median_area_difference_pct,mean_production_difference_pct,median_production_difference_pct,mean_yield_difference_pct,median_yield_difference_pct
0,Maize,552,5088.757143,38.082444,2701.436543,28.354995,8.821001,0.249004
1,Rice,485,111.758602,2.540731,272.771631,1.729400,6.703368,0.000000
2,Urad,488,1014.720453,90.181512,559.021099,100.000000,19.814259,0.452489
3,Wheat,537,25.149742,0.749780,22.277536,0.239218,0.000269,0.000000


In [ ]:
#Measuring how many observations agree within tolerances
tolerances = [1, 5, 10, 20]

for crop in validation_crops:

    crop_data = matched[
        matched["crop"] == crop
    ]

    print(crop)

    for tolerance in tolerances:

        area_pct = (
            crop_data[
                "area_difference_pct"
            ] <= tolerance
        ).mean() * 100

        production_pct = (
            crop_data[
                "production_difference_pct"
            ] <= tolerance
        ).mean() * 100

        yield_pct = (
            crop_data[
                "yield_difference_pct"
            ] <= tolerance
        ).mean() * 100

        print(
            f"Within {tolerance}% | "
            f"Area: {area_pct:.2f}% | "
            f"Production: {production_pct:.2f}% | "
            f"Yield: {yield_pct:.2f}%"
        )


Rice
Within 1% | Area: 38.97% | Production: 45.15% | Yield: 77.73%
Within 5% | Area: 58.14% | Production: 58.76% | Yield: 86.19%
Within 10% | Area: 63.51% | Production: 64.54% | Yield: 90.52%
Within 20% | Area: 67.63% | Production: 69.07% | Yield: 94.02%

Wheat
Within 1% | Area: 54.00% | Production: 61.45% | Yield: 100.00%
Within 5% | Area: 68.16% | Production: 71.51% | Yield: 100.00%
Within 10% | Area: 71.69% | Production: 73.18% | Yield: 100.00%
Within 20% | Area: 74.86% | Production: 76.35% | Yield: 100.00%

Maize
Within 1% | Area: 7.61% | Production: 12.68% | Yield: 56.70%
Within 5% | Area: 19.75% | Production: 26.45% | Yield: 71.01%
Within 10% | Area: 27.72% | Production: 38.04% | Yield: 76.81%
Within 20% | Area: 38.04% | Production: 46.01% | Yield: 87.68%

Urad
Within 1% | Area: 8.40% | Production: 5.53% | Yield: 53.89%
Within 5% | Area: 21.52% | Production: 16.19% | Yield: 63.52%
Within 10% | Area: 29.30% | Production: 22.34% | Yield: 72.54%
Within 20% | Area: 35.66% | Producti

In [19]:
#Checking exact yeild agreement
yield_check = (
    matched
    .groupby("crop")
    .agg(
        observations=(
            "yield_difference",
            "size"
        ),
        exact_matches=(
            "yield_difference",
            lambda x: (x == 0).sum()
        ),
        nonzero_differences=(
            "yield_difference",
            lambda x: (x != 0).sum()
        )
    )
    .reset_index()
)

yield_check["exact_match_percentage"] = (
    yield_check["exact_matches"]
    /
    yield_check["observations"]
    * 100
)

display(yield_check)

,crop,observations,exact_matches,nonzero_differences,exact_match_percentage
0,Maize,552,258,294,46.739130
1,Rice,485,311,174,64.123711
2,Urad,488,224,264,45.901639
3,Wheat,537,535,2,99.627561


In [ ]:
#Finding the largest discrepancies
#Area
display(
    matched[
        [
            "crop",
            "state_des",
            "district_des",
            "area_ha_des",
            "area_ha_upag",
            "area_difference_pct"
        ]
    ]
    .sort_values(
        "area_difference_pct",
        ascending=False
    )
    .head(20)
)

,crop,state_des,district_des,area_ha_des,area_ha_upag,area_difference_pct
2908,Maize,Tamil Nadu,Thoothukkudi,2.0,50000.0,2.499900e+06
2954,Urad,Tamil Nadu,Mayiladuthurai,12.0,24000.0,1.999000e+05
2576,Maize,Andhra Pradesh,Dr. B.R. Ambedkar Konaseema,1.0,1000.0,9.990000e+04
2546,Urad,Andhra Pradesh,Srikakulam,30.0,21000.0,6.990000e+04
2404,Maize,Maharashtra,Gondia,3.0,2000.0,6.656667e+04
2578,Urad,Andhra Pradesh,Dr. B.R. Ambedkar Konaseema,2.0,1000.0,4.990000e+04
2950,Urad,Tamil Nadu,Tenkasi,116.0,28000.0,2.403793e+04
2524,Maize,Andhra Pradesh,Guntur,109.0,26000.0,2.375321e+04
3074,Urad,Telangana,Nagarkurnool,36.0,6000.0,1.656667e+04
2554,Urad,Andhra Pradesh,Vizianagaram,123.0,20000.0,1.616016e+04


In [21]:
#Production
display(
    matched[
        [
            "crop",
            "state_des",
            "district_des",
            "production_tonnes_des",
            "production_tonnes_upag",
            "production_difference_pct"
        ]
    ]
    .sort_values(
        "production_difference_pct",
        ascending=False
    )
    .head(20)
)

,crop,state_des,district_des,production_tonnes_des,production_tonnes_upag,production_difference_pct
2908,Maize,Tamil Nadu,Thoothukkudi,19.0,177000.0,931478.947368
2576,Maize,Andhra Pradesh,Dr. B.R. Ambedkar Konaseema,4.0,11000.0,274900.000000
2524,Maize,Andhra Pradesh,Guntur,455.0,324000.0,71108.791209
2954,Urad,Tamil Nadu,Mayiladuthurai,10.0,5000.0,49900.000000
2546,Urad,Andhra Pradesh,Srikakulam,32.0,15000.0,46775.000000
2404,Maize,Maharashtra,Gondia,9.0,4000.0,44344.444444
1361,Rice,Tripura,West Tripura,192.0,80000.0,41566.666667
1369,Rice,Tripura,Sepahijala,427.0,159000.0,37136.533958
2580,Maize,Andhra Pradesh,Eluru,1990.0,352000.0,17588.442211
2528,Maize,Andhra Pradesh,Krishna,305.0,52000.0,16949.180328


In [22]:
#Yield
display(
    matched[
        [
            "crop",
            "state_des",
            "district_des",
            "yield_kg_ha_des",
            "yield_kg_ha_upag",
            "yield_difference_pct"
        ]
    ]
    .sort_values(
        "yield_difference_pct",
        ascending=False
    )
    .head(20)
)

,crop,state_des,district_des,yield_kg_ha_des,yield_kg_ha_upag,yield_difference_pct
3006,Urad,Telangana,Nizamabad,38.0,1369.0,3502.631579
602,Urad,Uttar Pradesh,Ambedkar Nagar,326.0,1247.0,282.515337
738,Urad,Uttar Pradesh,Kannauj,294.0,1008.0,242.857143
1369,Rice,Tripura,Sepahijala,1024.0,3480.0,239.843750
1357,Rice,Tripura,South Tripura,1046.0,3381.0,223.231358
1373,Rice,Tripura,Gomati,1052.0,3340.0,217.490494
1361,Rice,Tripura,West Tripura,1091.0,3437.0,215.032081
1377,Rice,Tripura,Unakoti,1040.0,3180.0,205.769231
2524,Maize,Andhra Pradesh,Guntur,4174.0,12491.0,199.257307
1365,Rice,Tripura,Khowai,1147.0,3197.0,178.727114


In [23]:
#Exporting the results
from pathlib import Path

VALIDATION_DIR = Path(
    "../data/processed/validation"
)

VALIDATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [24]:
validation_report.to_csv(
    VALIDATION_DIR
    / "des_upag_validation_summary.csv",
    index=False
)

match_summary.to_csv(
    VALIDATION_DIR
    / "des_upag_match_summary.csv",
    index=False
)

matched.to_csv(
    VALIDATION_DIR
    / "des_upag_matched_2022_23.csv",
    index=False
)